# 38 — Structured Outputs & Validation

### Why this exists
Valid JSON is not the same as valid data. This lab moves through syntax validation, schema validation and semantic validation, then makes repair bounded and observable.

### Learning objectives
parse JSON; validate schema; reject semantic contradictions; count repair attempts

### Prerequisite
Python json and dictionaries

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/38-structured-outputs-validation.svg)

> **Question to carry through the lab:** What should happen when a model returns syntactically valid JSON with the wrong type or unsafe value?

## Capability contract

**Capability:** C08 — LLM Systems  
**Workstream:** AI Support Platform  
**Primary roles:** AI Engineer|ML Engineer  
**You will prove:** Build or inspect a grounded AI mechanism


## Simulation → reality bridge

**Real-system bridge:** Structured-output model APIs

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook parses deterministic structured responses. Real model calls can produce malformed JSON, schema drift, partial output, refusals, truncation and semantically invalid values.

**Transfer challenge:** Design a provider-neutral contract test that distinguishes syntactic validity from semantic validity.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


## Mental model
Validation is a layered boundary: syntax → schema → semantics → policy. Repair is a controlled retry, not a substitute for validation.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S17 — 1190: Support assistant outputs cannot be validated consistently** → [`../work_simulations/17_acme_commerce_structured_support/README.md`](../work_simulations/17_acme_commerce_structured_support/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
import json
raw=[
 '{"priority": "high", "amount": 120}',
 '{"priority": "high", "amount": "120"}',
 '{"priority": "high", "amount": -5}'
]
for x in raw:
    try: print(json.loads(x))
    except json.JSONDecodeError as e: print("syntax failure",e)

In [ ]:
def schema_ok(x):
    return isinstance(x,dict) and isinstance(x.get("priority"),str) and isinstance(x.get("amount"),(int,float))
def semantic_ok(x):
    return isinstance(x.get("amount"),(int,float)) and x.get("amount")>=0 and x.get("priority") in {"low","medium","high"}
for x in raw:
    y=json.loads(x)
    print(y,schema_ok(y),semantic_ok(y))


In [ ]:
def bounded_repair(x,max_attempts=1):
    attempts=0
    if isinstance(x.get("amount"),str):
        attempts+=1; x["amount"]=float(x["amount"])
    return x,attempts
y,_=bounded_repair(json.loads(raw[1])); print(y,schema_ok(y),semantic_ok(y))

In [ ]:
bad,_=bounded_repair(json.loads(raw[2])); print("still rejected",schema_ok(bad) and semantic_ok(bad))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## Your task

**Goal:** Do not confuse valid JSON with valid output.

**Do this:** Add an unknown field and a policy-forbidden priority. Decide what can be repaired automatically and what must be rejected/escalated.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 8. Production bridge
Production structured-output pipelines should validate every boundary, cap retries, log raw/normalized outcomes safely, and never let “JSON parsed” imply correctness.

## 9. Independent challenge
Add an unknown field and a policy-forbidden priority. Decide which failures may be repaired and which must be rejected and surfaced to a human.

### Evidence to keep
syntax/schema/semantic results, bounded repair count, and rejection evidence

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

---

**Check your work:** [Open the reference solution](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/38.md) — only after you have attempted the exercise.